In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("medaliion").getOrCreate()
spark

In [14]:
spark.sql("SHOW CATALOGS").show()

+-------------+
|      catalog|
+-------------+
|    lakehouse|
|spark_catalog|
+-------------+



In [15]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|   bronze|
|   silver|
|     gold|
+---------+



In [17]:
# database 생성 (defualt = lakehouse)
spark.sql("create database if not exists bronze")
spark.sql("create database if not exists silver")
spark.sql("create database if not exists gold")

DataFrame[]

In [6]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+



In [6]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.bronze.daily_prices_raw (
    symbol STRING NOT NULL COMMENT '수집 요청 티커',
    date DATE NOT NULL COMMENT 'yfinance가 반환한 거래일',
    open DOUBLE NOT NULL,
    high DOUBLE NOT NULL,
    low DOUBLE NOT NULL,
    close DOUBLE NOT NULL,
    volume BIGINT NOT NULL,
    collected_at TIMESTAMP NOT NULL COMMENT '수집 시각',
    source STRING NOT NULL COMMENT '데이터 출처: yfinance',
    run_id STRING NOT NULL COMMENT '수집 실행 식별자'
)
USING iceberg
PARTITIONED BY (days(collected_at))
TBLPROPERTIES ('format-version' = '2');
""")

26/07/15 10:56:24 INFO RESTSessionCatalog: Table properties set at catalog level through catalog properties: {}
26/07/15 10:56:24 INFO RESTSessionCatalog: Table properties enforced at catalog level through catalog properties: {}


DataFrame[]

In [16]:
spark.sql("SHOW TABLES IN lakehouse.bronze").show()

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|   bronze|daily_prices_raw|      false|
+---------+----------------+-----------+



In [10]:
# 1. Silver 레이어에 daily_prices 테이블 생성 (먼저 실행)
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.silver.daily_prices (
    symbol STRING NOT NULL COMMENT '티커 심볼',
    date DATE NOT NULL COMMENT '거래일',
    open DOUBLE NOT NULL,
    high DOUBLE NOT NULL,
    low DOUBLE NOT NULL,
    close DOUBLE NOT NULL,
    volume BIGINT NOT NULL
)
USING iceberg
PARTITIONED BY (symbol)
COMMENT '정제된 일봉 주가 데이터 (중복 제거)';
""")

26/07/15 11:01:16 INFO RESTSessionCatalog: Table properties set at catalog level through catalog properties: {}
26/07/15 11:01:16 INFO RESTSessionCatalog: Table properties enforced at catalog level through catalog properties: {}


DataFrame[]

In [17]:
spark.sql("""
  MERGE INTO lakehouse.silver.daily_prices AS target
  USING (
    SELECT symbol, date, open, high, low, close, volume
    FROM (
      SELECT
        symbol,
        date,
        open,
        high,
        low,
        close,
        volume,
        ROW_NUMBER() OVER (
          PARTITION BY symbol, date
          ORDER BY collected_at DESC
        ) AS row_num
      FROM lakehouse.bronze.daily_prices_raw
    )
    WHERE row_num = 1
  ) AS source
  ON target.symbol = source.symbol
  AND target.date = source.date
  WHEN MATCHED THEN UPDATE SET
    target.open = source.open,
    target.high = source.high,
    target.low = source.low,
    target.close = source.close,
    target.volume = source.volume
  WHEN NOT MATCHED THEN INSERT (
    symbol, date, open, high, low, close, volume
  ) VALUES (
    source.symbol, source.date, source.open, source.high,
    source.low, source.close, source.volume
  );
""")

26/07/17 06:42:30 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table lakehouse.bronze.daily_prices_raw
26/07/17 06:42:30 INFO SnapshotScan: Scanning table lakehouse.silver.daily_prices snapshot 5876336397564974764 created at 2026-07-15T11:01:20.975+00:00 with filter true
26/07/17 06:42:30 INFO BaseDistributedDataScan: Planning file tasks locally for table lakehouse.silver.daily_prices
26/07/17 06:42:30 INFO LoggingMetricsReporter: Received metrics report: ScanReport{tableName=lakehouse.silver.daily_prices, snapshotId=5876336397564974764, filter=true, schemaId=0, projectedFieldIds=[1, 2147483646, 2, 3, 4, 5, 6, 7], projectedFieldNames=[symbol, _file, date, open, high, low, close, volume], scanMetrics=ScanMetricsResult{totalPlanningDuration=TimerResult{timeUnit=NANOSECONDS, totalDuration=PT0.03251725S, count=1}, resultDataFiles=CounterResult{unit=COUNT, value=0}, resultDeleteFiles=CounterResult{unit=COUNT, value=0}, totalDataManifests=CounterResu

DataFrame[]

In [18]:
spark.sql("SHOW TABLES IN lakehouse.silver").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|   silver|daily_prices|      false|
+---------+------------+-----------+

